# 02 — Full sweep (resumable across sessions)

**Run only after notebook 01 passes.**

A Kaggle session has a time limit, and only `/kaggle/working` survives — and
only if the run finishes normally. So the sweep stops itself after
`BUDGET_HOURS`, cleanly, and continues in the next session:

1. **Save Version → Save & Run All (Commit)** — runs in the background.
2. When it finishes, open the notebook, **Add Input** → *Notebook output*,
   and pick the version that just ran.
3. Commit again. Completed stages are restored and never re-run.
4. Repeat until the status table shows everything complete.

Settings: GPU T4 x2, Internet On, secrets `HF_TOKEN` and `ANTHROPIC_API_KEY`
(grading calls the API and uses credits), input: the project dataset.

In [ ]:
import os, glob, shutil, subprocess

# ---- Where the project comes from (pick ONE) -------------------------------
# A. Kaggle Dataset — recommended. Upload offline-health-ai-assist.zip as a new
#    dataset (Kaggle unpacks it), then "Add Input" -> your dataset. Edit nothing.
# B. GitHub — set GITHUB_URL. Private repository: add a GITHUB_TOKEN secret.
GITHUB_URL = ""   # e.g. "https://github.com/<you>/offline-health-ai-assist.git"
# -----------------------------------------------------------------------------
WORK = "/kaggle/working/repo"


def find_root(d):
    """Project root = the folder containing src/train.py, at any depth. Handles
    uploads that gained an extra top-level folder."""
    hits = glob.glob(os.path.join(d, "**", "src", "train.py"), recursive=True)
    hits.sort(key=lambda p: p.count(os.sep))
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None


def fresh_copy(fill):
    """Replace the code in WORK but keep progress (runs/, results/) from an
    earlier run in this session."""
    keep = "/kaggle/working/_keep"
    shutil.rmtree(keep, ignore_errors=True)
    old = find_root(WORK) if os.path.isdir(WORK) else None
    if old:
        for d in ("runs", "results"):
            if os.path.isdir(os.path.join(old, d)):
                shutil.move(os.path.join(old, d), os.path.join(keep, d))
    shutil.rmtree(WORK, ignore_errors=True)
    fill()
    root = find_root(WORK)
    if root and os.path.isdir(keep):
        for d in os.listdir(keep):
            shutil.move(os.path.join(keep, d), os.path.join(root, d))
    return root


dataset_root = next((r for r in (find_root(d) for d in sorted(glob.glob("/kaggle/input/*"))) if r), None)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

if dataset_root:
    source = f"Kaggle dataset ({dataset_root})"
    root = fresh_copy(lambda: shutil.copytree(dataset_root, WORK))   # inputs are read-only
elif GITHUB_URL:
    source = "GitHub"
    url = GITHUB_URL.replace("https://", f"https://{token}@") if token else GITHUB_URL
    def _clone():
        r = subprocess.run(["git", "clone", "--depth", "1", url, WORK], capture_output=True, text=True)
        if r.returncode:
            raise SystemExit("git clone failed:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
        # never leave a token in .git/config: /kaggle/working is saved with the output
        subprocess.run(["git", "-C", WORK, "remote", "set-url", "origin", GITHUB_URL])
    root = fresh_copy(_clone)
else:
    raise SystemExit("No project found. Attach the project dataset (Add Input), or set GITHUB_URL.")

if not root:
    raise SystemExit(f"{source} contains no src/train.py — the upload is incomplete.")
os.chdir(root)
for f in ("run_smoke_test.sh", "mobile/sync_rules.sh"):
    if os.path.exists(f):
        os.chmod(f, 0o755)
print(f"source: {source}\nproject root: {root}\n")

# Completeness and integrity BEFORE anything else. --strict also rejects files
# that merely DIFFER from the manifest: this copy came from the project zip, so
# a difference means a broken copy, and a broken copy fails later in ways that
# point nowhere near the cause. One altered data file previously made all six
# training arms fail identically, seconds in, before any of them saw the GPU.
r = subprocess.run(["python3", "scripts/verify_repo.py", "--strict"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode:
    raise SystemExit("This copy of the project is incomplete or altered (list above). "
                     "Upload the full zip as a Kaggle Dataset — see docs/KAGGLE.md.")

In [ ]:
for name in ("HF_TOKEN", "ANTHROPIC_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ[name] = UserSecretsClient().get_secret(name)
        print(f"{name}: set")
    except Exception:
        print(f"{name}: MISSING — " + ("Gemma will not download" if name == "HF_TOKEN"
                                     else "grading will fail"))

In [ ]:
# Pinned to the exact versions the pipeline was tested with. torch is NOT
# reinstalled: Kaggle ships a CUDA build matched to its drivers.
!pip install -q "transformers==5.17.0" "trl==1.13.0" "peft==0.19.1" "accelerate==1.13.0" "bitsandbytes==0.50.2" datasets pyyaml safetensors anthropic

In [ ]:
# Pick the GPU with the most free memory, and refuse to start if something is
# already holding memory. Every arm in the first attempt failed with "CUDA out
# of memory" within seconds — the QLoRA arm could not get even 20 MB — which
# means the GPU was occupied BEFORE training began. The usual cause is a model
# loaded in a cell of this notebook: it stays on the GPU until the kernel
# restarts, and every training run then competes with it.
try:
    q = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free,memory.total",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit("No GPU in this session. Settings -> Accelerator -> GPU T4 x2, then restart.")
gpus = []
for line in q.stdout.strip().splitlines():
    i, name, used, free, total = (x.strip() for x in line.split(","))
    gpus.append({"i": int(i), "name": name, "used": int(used), "free": int(free), "total": int(total)})
print(f"{'GPU':<5}{'name':<12}{'used MiB':>10}{'free MiB':>10}")
for g in gpus:
    print(f"{g['i']:<5}{g['name']:<12}{g['used']:>10}{g['free']:>10}")

if not gpus:
    raise SystemExit("nvidia-smi reported no GPUs. Settings -> Accelerator -> GPU T4 x2.")
best = max(gpus, key=lambda g: g["free"])
if best["free"] < 12000:
    print("\nNot enough free GPU memory to train. Something is already using it —")
    print("most likely a model loaded in an earlier cell of this notebook.")
    print("Fix: Run -> Restart & clear cell outputs, then run the cells in order,")
    print("without adding cells that load a model.")
    raise SystemExit(f"GPU {best['i']} has only {best['free']} MiB free (need 12000).")

os.environ["CUDA_VISIBLE_DEVICES"] = str(best["i"])        # inherited by every subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"\ntraining will use GPU {best['i']} with {best['free']} MiB free")

## Restore progress from the previous session's output, if attached

In [ ]:
import json
prev = [d for d in glob.glob("/kaggle/input/*")
        if glob.glob(os.path.join(d, "**", "runs", "pipeline_state.json"), recursive=True)]
if prev:
    state = sorted(glob.glob(os.path.join(prev[0], "**", "runs", "pipeline_state.json"),
                             recursive=True), key=len)[0]
    prev_root = os.path.dirname(os.path.dirname(state))
    for d in ("runs", "results"):
        if os.path.isdir(os.path.join(prev_root, d)):
            shutil.copytree(os.path.join(prev_root, d), d, dirs_exist_ok=True)
    done = len(json.load(open("runs/pipeline_state.json")).get("done", {}))
    print(f"restored from {prev_root}: {done} stages already complete")
else:
    print("fresh start — no previous output attached")

## The data must exist, and must not leak

In [ ]:
for f in ("data/train.jsonl", "data/val.jsonl", "data/test.jsonl"):
    n = sum(1 for _ in open(f)) if os.path.exists(f) else 0
    print(f"{f:<22}{'MISSING' if not n else f'{n} rows'}")
if not (os.path.exists("data/train.jsonl") and os.path.exists("data/test.jsonl")):
    raise SystemExit("Build data/train.jsonl, val.jsonl and test.jsonl first "
                     "(data/build_dataset.py), then re-upload the project.")
r = subprocess.run(["python3", "data/check_leakage.py", "data/train.jsonl", "data/test.jsonl"],
                   capture_output=True, text=True)
print("\n".join(r.stdout.strip().splitlines()[-4:]))
if r.returncode:
    raise SystemExit("Train/test leakage — a model that has seen test questions scores from memory.")

In [ ]:
!python3 src/run_pipeline.py --status | tail -30

## Run until done or out of budget

In [ ]:
BUDGET_HOURS = 7.0   # well inside the session limit, so outputs are always saved
!python3 src/run_pipeline.py --stop-on-fail --budget-hours {BUDGET_HOURS}

In [ ]:
!python3 src/run_pipeline.py --status | tail -40